# Search similar MS/MS spectra

[Open in Colab](https://colab.research.google.com/github/Dsadd4/UltraMS/blob/main/cookbook/tutorials/spectrum_search.ipynb)

Rank the other spectra in the same example file by cosine similarity using the UltraMS Search model. The input file is the [five-spectrum DreaMS example](../../examples/data/NOTICE.md). Run cells from top to bottom.

## Install

In [ ]:
%pip install -q ultrams


## Read spectra

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import torch
from ultrams import UltraMS, read_spectra

sample = Path("examples/data/example_5_spectra.mgf")
if not sample.exists():
    sample = Path("example_5_spectra.mgf")
    if not sample.exists():
        urlretrieve(
            "https://raw.githubusercontent.com/Dsadd4/UltraMS/main/examples/data/example_5_spectra.mgf",
            sample,
        )
spectra = list(read_spectra(sample))
assert len(spectra) == 5
print(f"Loaded {len(spectra)} MS/MS spectra")

## Encode and rank

This small in-file ranking illustrates the Search representation. It is not an UltraAtlas search or a retrieval benchmark.

In [ ]:
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
model = UltraMS.from_pretrained("search", device=device)
embeddings = model.encode_batch(spectra, batch_size=5)
unit = embeddings / np.maximum(np.linalg.norm(embeddings, axis=1, keepdims=True), 1e-12)
query_index = 0
scores = unit @ unit[query_index]
ranked = [index for index in np.argsort(-scores) if index != query_index]
print("Query:", spectra[query_index]["id"])
for rank, index in enumerate(ranked, start=1):
    print(rank, spectra[index]["id"], float(scores[index]))